<style>
h1, h1 *, 
h2, h2 *, 
h3, h3 *, 
h4, h4 *, 
h5, h5 *, 
h6, h6 * {
  font-family: "Times New Roman", Times, serif !important;
}
</style>

## **Optimising Marketing with Artificial Intelligence (OMAI)**

<style>
h1, h1 *, 
h2, h2 *, 
h3, h3 *, 
h4, h4 *, 
h5, h5 *, 
h6, h6 * {
  font-family: "Times New Roman", Times, serif !important;
}
</style>

### **Transformer-Based Sentiment Analysis Models - DistilBERT (Round 2)**

---

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

OMAI_distilbert_round2 = pd.read_csv('OMAI - Data (TBSAMs) (Round 2).csv')
OMAI_distilbert_round2 = OMAI_distilbert_round2.drop('Unnamed: 0', axis = 1)
OMAI_distilbert_round2.iloc[0:1]

In [ ]:
import os
current_folder = os.getcwd()
print(current_folder)

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import transformers
import os
import shutil
import tarfile
import pandas as pd
import re
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.offline as pyo
import plotly.graph_objects as go
from wordcloud import WordCloud, STOPWORDS
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModel, BertTokenizerFast
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from bs4 import BeautifulSoup
from transformers import BertTokenizer, TFBertForSequenceClassification

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
OMAI_distilbert_round2['sentiment_score_nn'].value_counts(normalize = True)

sentiment_score_nn
2    0.720865
0    0.176512
1    0.102624
Name: proportion, dtype: float64

In [ ]:
from sklearn.model_selection import train_test_split

def data_splits12(OMAI_distilbert_round2): 
    train_df12, temp_df12 = train_test_split(OMAI_distilbert_round2, test_size = 0.2,
                                             stratify = OMAI_distilbert_round2['sentiment_score_nn'], random_state = 1500)

    val_df12, test_df12 = train_test_split(temp_df12, test_size = 0.5, 
                                           stratify = temp_df12['sentiment_score_nn'], random_state = 1500)

    return (train_df12.reset_index(drop = True), val_df12.reset_index(drop = True), test_df12.reset_index(drop = True))

train_df12, val_df12, test_df12 = data_splits12(OMAI_distilbert_round2)

train_text12 = train_df12['combined_text']
train_labels12 = train_df12['sentiment_score_nn']

val_text12 = val_df12['combined_text']
val_labels12 = val_df12['sentiment_score_nn']

test_text12 = test_df12['combined_text']
test_labels12 = test_df12['sentiment_score_nn']

In [ ]:
from transformers import AutoModel, BertTokenizerFast

bert12 = AutoModel.from_pretrained('distilbert-base-cased')
tokenizer12 = BertTokenizerFast.from_pretrained('distilbert-base-cased')

In [ ]:
seq_len12 = [len(i.split()) for i in train_text12]
pd.Series(seq_len12).hist(bins = 20)

In [ ]:
import psutil
import os
import torch
import numpy as np
from sklearn.model_selection import train_test_split

def print_memory_usage12(note = ""):
    process = psutil.Process(os.getpid())
    mem = process.memory_info().rss / (1024 ** 3)
    print(f"[{note}] Memory usage: {mem:.2f} GB")

def batch_tokenize12(texts12, tokenizer12, max_length = 128, tokenize_batch_size12 = 10000):
    input_ids12, attention_masks12 = [], []
    print_memory_usage12("Start batch_tokenize12")

    for i in range(0, len(texts12), tokenize_batch_size12):
        print(f"  → Tokenizing batch {i}–{i + tokenize_batch_size12}")
        batch12 = tokenizer12.batch_encode_plus(
            texts12[i:i + tokenize_batch_size12],
            max_length = max_length,
            padding = 'max_length',
            truncation = True,
            return_tensors = 'pt')
        input_ids12.append(batch12['input_ids'])
        attention_masks12.append(batch12['attention_mask'])
        print_memory_usage12(f"After batch {i}–{i + tokenize_batch_size12}")

    all_input_ids12 = torch.cat(input_ids12, dim = 0)
    all_attention_masks12 = torch.cat(attention_masks12, dim = 0)

    print_memory_usage12("End batch_tokenize12")
    return all_input_ids12, all_attention_masks12

def sample_10_percent_df12(input_df12):
    _, sampled_df12 = train_test_split(input_df12, test_size = 0.10,
                                      stratify = input_df12['sentiment_score_nn'], random_state = np.random.randint(10000))
    return sampled_df12.reset_index(drop = True)

train_df_small12 = sample_10_percent_df12(train_df12)
val_df_small12 = sample_10_percent_df12(val_df12)
test_df_small12 = sample_10_percent_df12(test_df12)

train_seq12, train_mask12 = batch_tokenize12(train_df_small12['combined_text'].tolist(), tokenizer12)
val_seq12, val_mask12 = batch_tokenize12(val_df_small12['combined_text'].tolist(), tokenizer12)
test_seq12, test_mask12 = batch_tokenize12(test_df_small12['combined_text'].tolist(), tokenizer12)

print_memory_usage12("Before label tensor conversion")
train_y12 = torch.tensor(train_df_small12['sentiment_score_nn'].tolist())
val_y12 = torch.tensor(val_df_small12['sentiment_score_nn'].tolist())
test_y12 = torch.tensor(test_df_small12['sentiment_score_nn'].tolist())
print_memory_usage12("After label tensor conversion")

In [10]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

batch_size12 = 16

train_data12 = TensorDataset(train_seq12, train_mask12, train_y12)
val_data12   = TensorDataset(val_seq12, val_mask12, val_y12)
test_data12  = TensorDataset(test_seq12, test_mask12, test_y12)

train_sampler12 = RandomSampler(train_data12)
val_sampler12   = SequentialSampler(val_data12)
test_sampler12  = SequentialSampler(test_data12)

train_dataloader12 = DataLoader(train_data12, sampler = train_sampler12, batch_size = batch_size12)
val_dataloader12   = DataLoader(val_data12, sampler = val_sampler12, batch_size = batch_size12)
test_dataloader12  = DataLoader(test_data12, sampler = test_sampler12, batch_size = batch_size12)

for param in bert12.parameters():
    param.requires_grad = False

In [11]:
import torch.nn as nn

class BERT_Arch12(nn.Module):  
    def __init__(self, bert12):
        super(BERT_Arch12, self).__init__()

        self.bert = bert12
        self.dropout = nn.Dropout(0.1)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(768, 512)
        self.fc2 = nn.Linear(512, 3)

    def forward(self, sent_id, mask):
        outputs = self.bert(sent_id, attention_mask = mask, return_dict = True)
        cls_hs = outputs.last_hidden_state[:, 0, :] 

        x = self.fc1(cls_hs)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x

model12 = BERT_Arch12(bert12)
model12 = model12.to(device)

In [12]:
from torch.optim import AdamW
from sklearn.utils.class_weight import compute_class_weight

optimizer12 = AdamW(model12.parameters(), lr = 2e-5, weight_decay = 0.01)

class_weights12 = compute_class_weight(class_weight = 'balanced', classes = np.unique(train_df_small12['sentiment_score_nn']),
                                       y = train_df_small12['sentiment_score_nn'])

weights12 = torch.tensor(class_weights12, dtype = torch.float).to(device)

cross_entropy12 = nn.CrossEntropyLoss(weight = weights12)

epochs12 = 5

In [13]:
def train12():
    model12.train()
    total_loss12, total_accuracy12 = 0, 0
    total_preds12 = []

    for step, batch in enumerate(train_dataloader12):
        if step % 50 == 0 and not step == 0:
            print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(train_dataloader12)))

        batch = [r.to(device) for r in batch]
        sent_id, mask, labels = batch

        model12.zero_grad()

        preds12 = model12(sent_id, mask)
        loss12 = cross_entropy12(preds12, labels)

        total_loss12 += loss12.item()

        loss12.backward()
        torch.nn.utils.clip_grad_norm_(model12.parameters(), 1.0)
        optimizer12.step()

        preds12 = preds12.detach().cpu().numpy()
        total_preds12.append(preds12)

    avg_loss12 = total_loss12 / len(train_dataloader12)

    if len(total_preds12) == 0:
        print("Warning: train12() produced no predictions. Check your training dataloader.")

    total_preds12 = np.concatenate(total_preds12, axis = 0)

    return avg_loss12, total_preds12

In [14]:
import datetime
import time

def format_time12(elapsed):
    return str(datetime.timedelta(seconds = int(round(elapsed))))

def evaluate12():
    print("\nEvaluating...")
    t0 = time.time()

    model12.eval()
    total_loss12, total_accuracy12 = 0, 0
    total_preds12 = []

    for step, batch in enumerate(val_dataloader12):
        if step % 50 == 0 and not step == 0:
            elapsed = format_time12(time.time() - t0)
            print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(val_dataloader12)))

        batch = [t.to(device) for t in batch]
        sent_id, mask, labels = batch

        with torch.no_grad():
            preds12 = model12(sent_id, mask)
            loss12 = cross_entropy12(preds12, labels)

            total_loss12 += loss12.item()
            preds12 = preds12.detach().cpu().numpy()
            total_preds12.append(preds12)

    avg_loss12 = total_loss12 / len(val_dataloader12)

    if len(total_preds12) == 0:
        print("Warning: evaluate12() produced no predictions. Check your validation dataloader.")

    total_preds12 = np.concatenate(total_preds12, axis = 0)

    return avg_loss12, total_preds12

In [15]:
best_valid_loss12 = float('inf')

epochs12 = 5

patience12 = 2
patience_counter12 = 0

train_losses12 = []
valid_losses12 = []

for epoch in range(epochs12):
    print('\n Epoch {:} / {:}'.format(epoch + 1, epochs12))
    print_memory_usage12(f"Start of Epoch {epoch + 1}")

    print_memory_usage12("Before training")
    train_loss12, _ = train12()
    print_memory_usage12("After training")

    print_memory_usage12("Before validation")
    valid_loss12, _ = evaluate12()
    print_memory_usage12("After validation")

    if valid_loss12 < best_valid_loss12:
        best_valid_loss12 = valid_loss12
        patience_counter12 = 0
        print_memory_usage12("Before saving model weights")
        torch.save(model12.state_dict(), 'DistilBERT_bestweights12.pt')
        print_memory_usage12("After saving model weights")
    else:
        patience_counter12 += 1
        if patience_counter12 >= patience12:
            print(f"\nEarly stopping triggered after {epoch + 1} epochs.")
            break

    train_losses12.append(train_loss12)
    valid_losses12.append(valid_loss12)

    print(f'\nTraining Loss: {train_loss12:.3f}')
    print(f'Validation Loss: {valid_loss12:.3f}')
    print_memory_usage12(f"End of Epoch {epoch + 1}")


 Epoch 1 / 5
[Start of Epoch 1] Memory usage: 1.26 GB
[Before training] Memory usage: 1.26 GB
  Batch    50  of  1,243.
  Batch   100  of  1,243.
  Batch   150  of  1,243.
  Batch   200  of  1,243.
  Batch   250  of  1,243.
  Batch   300  of  1,243.
  Batch   350  of  1,243.
  Batch   400  of  1,243.
  Batch   450  of  1,243.
  Batch   500  of  1,243.
  Batch   550  of  1,243.
  Batch   600  of  1,243.
  Batch   650  of  1,243.
  Batch   700  of  1,243.
  Batch   750  of  1,243.
  Batch   800  of  1,243.
  Batch   850  of  1,243.
  Batch   900  of  1,243.
  Batch   950  of  1,243.
  Batch 1,000  of  1,243.
  Batch 1,050  of  1,243.
  Batch 1,100  of  1,243.
  Batch 1,150  of  1,243.
  Batch 1,200  of  1,243.
[After training] Memory usage: 0.52 GB
[Before validation] Memory usage: 0.52 GB

Evaluating...
  Batch    50  of    156.
  Batch   100  of    156.
  Batch   150  of    156.
[After validation] Memory usage: 0.52 GB
[Before saving model weights] Memory usage: 0.52 GB
[After saving 

In [ ]:
from sklearn.metrics import (classification_report, accuracy_score, 
                             confusion_matrix, roc_auc_score, roc_curve)

from sklearn.preprocessing import label_binarize
from torch.utils.data import TensorDataset, DataLoader, SequentialSampler
import numpy as np

all_metrics12 = []          
all_fpr_tpr_auc12 = []      

def get_predictions12(model12, dataloader12):
    model12.eval()
    all_preds12, all_labels12 = [], []

    for batch in dataloader12:
        batch = [t.to(device) for t in batch]
        sent_id, mask, labels = batch

        with torch.no_grad():
            logits12 = model12(sent_id, mask)
            preds12 = torch.argmax(logits12, dim = 1)

        all_preds12.extend(preds12.cpu().numpy())
        all_labels12.extend(labels.cpu().numpy())

    return all_labels12, all_preds12

def get_pred_probs12(model12, dataloader12):
    model12.eval()
    all_probs12, all_labels12 = [], []

    for batch in dataloader12:
        batch = [t.to(device) for t in batch]
        sent_id, mask, labels = batch

        with torch.no_grad():
            logits12 = model12(sent_id, mask)
            probs12 = torch.softmax(logits12, dim = 1)

        all_probs12.extend(probs12.cpu().numpy())
        all_labels12.extend(labels.cpu().numpy())

    return np.array(all_labels12), np.array(all_probs12)

train_true_labels12, train_preds12 = get_predictions12(model12, train_dataloader12)
print("Training Classification Report:")
print(classification_report(train_true_labels12, train_preds12))
print("Training Accuracy:", accuracy_score(train_true_labels12, train_preds12))

val_true_labels12, val_preds12 = get_predictions12(model12, val_dataloader12)
print("\nValidation Classification Report:")
print(classification_report(val_true_labels12, val_preds12))
print("Validation Accuracy:", accuracy_score(val_true_labels12, val_preds12))

test_true_labels12, test_preds12 = get_predictions12(model12, test_dataloader12)
print("\nTest Classification Report:")
print(classification_report(test_true_labels12, test_preds12))
print("Test Accuracy:", accuracy_score(test_true_labels12, test_preds12))

Training Classification Report:
              precision    recall  f1-score   support

           0       0.47      0.71      0.56      3508
           1       0.24      0.34      0.28      2039
           2       0.90      0.73      0.81     14326

    accuracy                           0.69     19873
   macro avg       0.53      0.59      0.55     19873
weighted avg       0.76      0.69      0.71     19873

Training Accuracy: 0.6861570975695668

Validation Classification Report:
              precision    recall  f1-score   support

           0       0.45      0.69      0.54       438
           1       0.26      0.35      0.30       255
           2       0.90      0.73      0.80      1791

    accuracy                           0.68      2484
   macro avg       0.53      0.59      0.55      2484
weighted avg       0.75      0.68      0.71      2484

Validation Accuracy: 0.6831723027375202

Test Classification Report:
              precision    recall  f1-score   support

         

In [ ]:
import pandas as pd
from sklearn.preprocessing import label_binarize
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_curve, roc_auc_score

train_acc12 = accuracy_score(train_true_labels12, train_preds12)
val_acc12 = accuracy_score(val_true_labels12, val_preds12)
test_acc12 = accuracy_score(test_true_labels12, test_preds12)

conf_matrix_train12 = confusion_matrix(train_true_labels12, train_preds12)
conf_matrix_val12 = confusion_matrix(val_true_labels12, val_preds12)
conf_matrix_test12 = confusion_matrix(test_true_labels12, test_preds12)

def print_conf_matrix12(cm12, title12):
    print(f"\n{title12} Confusion Matrix - Model 1 Round 2 (12):")
    df12 = pd.DataFrame(cm12,
                        index = ["True Negative", "True Neutral", "True Positive"],
                        columns = ["Predicted Negative", "Predicted Neutral", "Predicted Positive"])
    print(df12)

print_conf_matrix12(conf_matrix_train12, "Training")
print_conf_matrix12(conf_matrix_val12, "Validation")
print_conf_matrix12(conf_matrix_test12, "Test")

train_probs_true12, train_probs12 = get_pred_probs12(model12, train_dataloader12)
val_probs_true12, val_probs12 = get_pred_probs12(model12, val_dataloader12)
test_probs_true12, test_probs12 = get_pred_probs12(model12, test_dataloader12)

def get_roc_data12(y_true12, y_probs12, set_name12, model_round12 = '12'):
    y_bin12 = label_binarize(y_true12, classes = [0, 1, 2])
    roc_list12 = []
    for i in range(3):
        fpr12, tpr12, _ = roc_curve(y_bin12[:, i], y_probs12[:, i])
        auc12 = roc_auc_score(y_bin12[:, i], y_probs12[:, i])
        roc_list12.append({
            'model_round': model_round12,
            'set': set_name12,
            'class': i,
            'fpr': fpr12,
            'tpr': tpr12,
            'auc': auc12})
    return roc_list12

roc_train12 = get_roc_data12(train_probs_true12, train_probs12, set_name12 = 'train')
roc_val12 = get_roc_data12(val_probs_true12, val_probs12, set_name12 = 'val')
roc_test12 = get_roc_data12(test_probs_true12, test_probs12, set_name12 = 'test')

round_metrics12 = {
    'model_round': '12',
    'train_accuracy': train_acc12,
    'val_accuracy': val_acc12,
    'test_accuracy': test_acc12,
    'train_report': classification_report(train_true_labels12, train_preds12, output_dict = True),
    'val_report': classification_report(val_true_labels12, val_preds12, output_dict = True),
    'test_report': classification_report(test_true_labels12, test_preds12, output_dict = True),
    'conf_matrix_train12': conf_matrix_train12,
    'conf_matrix_val12': conf_matrix_val12,
    'conf_matrix_test12': conf_matrix_test12}

all_metrics12.append(round_metrics12)

all_fpr_tpr_auc12.append({
    'model_round': '12',
    'roc_train12': roc_train12,
    'roc_val12': roc_val12,
    'roc_test12': roc_test12})


Training Confusion Matrix - Model 1 Round 2 (12):
               Predicted Negative  Predicted Neutral  Predicted Positive
True Negative                2480                532                 496
True Neutral                  678                694                 667
True Positive                2164               1700               10462

Validation Confusion Matrix - Model 1 Round 2 (12):
               Predicted Negative  Predicted Neutral  Predicted Positive
True Negative                 301                 69                  68
True Neutral                   81                 89                  85
True Positive                 294                190                1307

Test Confusion Matrix - Model 1 Round 2 (12):
               Predicted Negative  Predicted Neutral  Predicted Positive
True Negative                 323                 54                  62
True Neutral                  103                 72                  80
True Positive                 265             

In [ ]:
from pprint import pprint

print("Stored Metrics for Model 1 Round 2")
pprint(all_metrics12[-1])  

==== Stored Metrics for Model 1 Round 2 ====
{'conf_matrix_test12': array([[ 323,   54,   62],
       [ 103,   72,   80],
       [ 265,  199, 1327]], dtype=int64),
 'conf_matrix_train12': array([[ 2480,   532,   496],
       [  678,   694,   667],
       [ 2164,  1700, 10462]], dtype=int64),
 'conf_matrix_val12': array([[ 301,   69,   68],
       [  81,   89,   85],
       [ 294,  190, 1307]], dtype=int64),
 'model_round': '12',
 'test_accuracy': 0.6929577464788732,
 'test_report': {'0': {'f1-score': 0.5716814159292035,
                       'precision': 0.467438494934877,
                       'recall': 0.7357630979498861,
                       'support': 439.0},
                 '1': {'f1-score': 0.2482758620689655,
                       'precision': 0.22153846153846155,
                       'recall': 0.2823529411764706,
                       'support': 255.0},
                 '2': {'f1-score': 0.8141104294478527,
                       'precision': 0.9033356024506467,
      

In [ ]:
print("\nAccuracies:")
print("Training:", all_metrics12[-1]['train_accuracy'])
print("Validation:  ", all_metrics12[-1]['val_accuracy'])
print("Test: ", all_metrics12[-1]['test_accuracy'])

In [21]:
print("\nTest Confusion Matrix:")
print(all_metrics12[-1]['conf_matrix_test12'])


Test Confusion Matrix:
[[ 323   54   62]
 [ 103   72   80]
 [ 265  199 1327]]


In [22]:
print("\nTraining Confusion Matrix:")
print(all_metrics12[-1]['conf_matrix_train12'])


Training Confusion Matrix:
[[ 2480   532   496]
 [  678   694   667]
 [ 2164  1700 10462]]


In [23]:
print("\nValidation Confusion Matrix:")
print(all_metrics12[-1]['conf_matrix_val12'])


Validation Confusion Matrix:
[[ 301   69   68]
 [  81   89   85]
 [ 294  190 1307]]


In [25]:
print("\nAUC Scores by Class for Test Set:")
for entry in all_fpr_tpr_auc12[-1]['roc_test12']:
    print(f"Class {entry['class']} - AUC: {entry['auc']:.4f}")


AUC Scores by Class for Test Set:
Class 0 - AUC: 0.8649
Class 1 - AUC: 0.7212
Class 2 - AUC: 0.8470


In [26]:
print("\nAUC Scores by Class for Training Set:")
for entry in all_fpr_tpr_auc12[-1]['roc_train12']:
    print(f"Class {entry['class']} - AUC: {entry['auc']:.4f}")


AUC Scores by Class for Training Set:
Class 0 - AUC: 0.8617
Class 1 - AUC: 0.7295
Class 2 - AUC: 0.8405


In [27]:
print("\nAUC Scores by Class for Validation Set:")
for entry in all_fpr_tpr_auc12[-1]['roc_val12']:
    print(f"Class {entry['class']} - AUC: {entry['auc']:.4f}")


AUC Scores by Class for Validation Set:
Class 0 - AUC: 0.8560
Class 1 - AUC: 0.7141
Class 2 - AUC: 0.8377


In [29]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['font.family'] = 'Georgia'

def roc_data12(class_id, roc_data12):
    for entry12 in roc_data12: 
        if entry12['class'] == class_id:
            return entry12['fpr'], entry12['tpr'], entry12['auc']
    return None, None, None

roc_entry12 = all_fpr_tpr_auc12[-1]
roc_train12 = roc_entry12['roc_train12']
roc_val12 = roc_entry12['roc_val12']
roc_test12 = roc_entry12['roc_test12']

In [ ]:
fpr_train12, tpr_train12, auc_train12 = roc_data12(0, roc_train12)
fpr_val12, tpr_val12, auc_val12 = roc_data12(0, roc_val12)
fpr_test12, tpr_test12, auc_test12 = roc_data12(0, roc_test12)

fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(fpr_train12, tpr_train12, label = f'Training (AUC = {auc_train12:.2f})', linewidth = 2)
ax.plot(fpr_val12, tpr_val12, label = f'Validation (AUC = {auc_val12:.2f})', linewidth = 2)
ax.plot(fpr_test12, tpr_test12, label = f'Test (AUC = {auc_test12:.2f})', linewidth = 2)
ax.plot([0, 1], [0, 1], 'k--', linewidth = 1)

ax.set_title("DistilBERT (Round 2) - ROC Curve for Class 0 (Negative)", fontweight = 'bold', fontsize = 14)
ax.set_xlabel("False Positive Rate (FPR)", fontsize = 12, fontweight = 'bold')
ax.set_ylabel("True Positive Rate (TPR)", fontsize = 12, fontweight = 'bold')
ax.tick_params(axis = 'both', labelsize = 10)
ax.grid(True, linestyle = '--', linewidth = 0.5, alpha = 0.7)
ax.legend(fontsize = 10)

plt.tight_layout()
plt.show()

In [ ]:
fpr_train12, tpr_train12, auc_train12 = roc_data12(1, roc_train12)
fpr_val12, tpr_val12, auc_val12 = roc_data12(1, roc_val12)
fpr_test12, tpr_test12, auc_test12 = roc_data12(1, roc_test12)

fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(fpr_train12, tpr_train12, label = f'Training (AUC = {auc_train12:.2f})', linewidth = 2)
ax.plot(fpr_val12, tpr_val12, label = f'Validation (AUC = {auc_val12:.2f})', linewidth = 2)
ax.plot(fpr_test12, tpr_test12, label = f'Test (AUC = {auc_test12:.2f})', linewidth = 2)
ax.plot([0, 1], [0, 1], 'k--', linewidth = 1)

ax.set_title("DistilBERT (Round 2) - ROC Curve for Class 1 (Neutral)", fontweight = 'bold', fontsize = 14)
ax.set_xlabel("False Positive Rate (FPR)", fontsize = 12, fontweight = 'bold')
ax.set_ylabel("True Positive Rate (TPR)", fontsize = 12, fontweight = 'bold')
ax.tick_params(axis = 'both', labelsize = 10)
ax.grid(True, linestyle = '--', linewidth = 0.5, alpha = 0.7)
ax.legend(fontsize = 10)

plt.tight_layout()
plt.show()

In [ ]:
fpr_train12, tpr_train12, auc_train12 = roc_data12(2, roc_train12)
fpr_val12, tpr_val12, auc_val12 = roc_data12(2, roc_val12)
fpr_test12, tpr_test12, auc_test12 = roc_data12(2, roc_test12)

fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(fpr_train12, tpr_train12, label = f'Training (AUC = {auc_train12:.2f})', linewidth = 2)
ax.plot(fpr_val12, tpr_val12, label = f'Validation (AUC = {auc_val12:.2f})', linewidth = 2)
ax.plot(fpr_test12, tpr_test12, label = f'Test (AUC = {auc_test12:.2f})', linewidth = 2)
ax.plot([0, 1], [0, 1], 'k--', linewidth = 1)

ax.set_title("DistilBERT (Round 2) - ROC Curve for Class 2 (Positive)", fontweight = 'bold', fontsize = 14)
ax.set_xlabel("False Positive Rate (FPR)", fontsize = 12, fontweight = 'bold')
ax.set_ylabel("True Positive Rate (TPR)", fontsize = 12, fontweight = 'bold')
ax.tick_params(axis = 'both', labelsize = 10)
ax.grid(True, linestyle = '--', linewidth = 0.5, alpha = 0.7)
ax.legend(fontsize = 10)

plt.tight_layout()
plt.show()

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, SequentialSampler

tokens_train_small12 = tokenizer12.batch_encode_plus(train_df_small12['combined_text'].tolist(),
                                                     max_length = 128,
                                                     padding = 'max_length',
                                                     truncation = True,
                                                     return_tensors = 'pt')

train_small_seq12 = tokens_train_small12['input_ids']
train_small_mask12 = tokens_train_small12['attention_mask']

train_small_data12 = TensorDataset(train_small_seq12, train_small_mask12)
train_small_dataloader12 = DataLoader(train_small_data12,
                                      sampler = SequentialSampler(train_small_data12),
                                      batch_size = 32)

model12.eval()
train_small_pred_labels12 = []
train_small_probs12 = []

for batch in train_small_dataloader12:
    batch = [t.to(device) for t in batch]
    sent_id, mask = batch

    with torch.no_grad():
        logits12 = model12(sent_id, mask)
        probs12 = torch.softmax(logits12, dim = 1)
        preds12 = torch.argmax(probs12, dim = 1)

    train_small_pred_labels12.extend(preds12.cpu().numpy())
    train_small_probs12.extend(probs12.cpu().numpy())

train_small_pred_labels12 = np.array(train_small_pred_labels12)
train_small_probs12 = np.array(train_small_probs12)

bert1_round2_df = train_df_small12.reset_index(drop = True).copy()
bert1_round2_df['bert_pred_label12'] = train_small_pred_labels12
bert1_round2_df['bert_pred_prob_neg12'] = train_small_probs12[:, 0]
bert1_round2_df['bert_pred_prob_neutral12'] = train_small_probs12[:, 1]
bert1_round2_df['bert_pred_prob_pos12'] = train_small_probs12[:, 2]

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

OMAI_distilbert_round2_output = bert1_round2_df.copy(deep = True)
OMAI_distilbert_round2_output.iloc[0:1]
print(OMAI_distilbert_round2_output.shape)

In [ ]:
OMAI_distilbert_round2_output.to_csv('OMAI - Results - DistilBERT (Round 2).csv')